In [9]:
import json
import logging
import os
import random
import socket
import uuid
from collections import defaultdict
from dataclasses import asdict

import slurminade
from dc_triangulation import (
    Graph_Wrapper,
    Ortools,
    Ortools_Parameter,
    Run_Algbench,
)

asdict
TIMEOUT = 300
path = os.path.join(
    "/home/fabian/uni/Bachelorarbeit/bachelorarbeit-fabian-alich/code/evaluation/eval#6",
    "instances",
)
NUMBER_RUNS = 5  # Number of runs for each instance
# This is the entry point for the evaluation script
# It will run the Run_Instance class from run_algbench module


def get_key_from_pos(pos):
    assert (isinstance(pos, tuple)) and len(pos) == 2, (
        "Position must be a list of two elements, bus is",
        pos,
    )
    return f"{pos[0]}_{pos[1]}"


def load_data():
    """Load data from calculated_data.json file"""
    calculated_data_file = os.path.join(
        "/home/fabian/uni/Bachelorarbeit/bachelorarbeit-fabian-alich/code/evaluation/eval#6", "calculated_data.json"
    )
    try:
        with open(calculated_data_file, "r") as f:
            data = json.load(f)
        logging.info(f"Loaded data from {calculated_data_file}")
        convertet_data = defaultdict(list)
        for item, value in data.items():
            for edge in value:
                assert isinstance(edge, list) and len(edge) == 2, (
                    "Each edge must be a list of two elements, but got",
                    edge,
                )
                assert isinstance(edge[0], list) and isinstance(edge[1], list), (
                    "Each edge must contain tuples, but got",
                    edge,
                )
                assert len(edge[0]) == 2 and len(edge[1]) == 2, (
                    "Each tuple in the edge must have two elements, but got",
                    edge,
                )
                convertet_data[item].append((tuple(edge[0]), tuple(edge[1])))
        return convertet_data
    except FileNotFoundError:
        logging.error(f"Could not find calculated_data.json at {calculated_data_file}")
        return {}
    except json.JSONDecodeError:
        logging.error(f"Could not parse JSON from {calculated_data_file}")
        return {}


data = load_data()


outer_parameter = {
    Ortools: [
        # {
        #     "timeout": TIMEOUT,
        #     "args": asdict(
        #         Ortools_Parameter(
        #             intersection=True,
        #             degree=True,
        #         )
        #     ),
        # },
        # {
        #     "timeout": TIMEOUT,
        #     "args": asdict(
        #         Ortools_Parameter(intersection=True, degree=True, maximize_edges=0.1)
        #     ),
        #     "hack_eval_6": True,
        #     "hack_eval_6_data": data,
        #     "hack_eval_6_PERCENT": 0.1,
        # },
        # {
        #     "timeout": TIMEOUT,
        #     "args": asdict(
        #         Ortools_Parameter(intersection=True, degree=True, maximize_edges=0.5)
        #     ),
        #     "hack_eval_6": True,
        #     "hack_eval_6_data": data,
        #     "hack_eval_6_PERCENT": 0.5,
        # },
        # {
        #     "timeout": TIMEOUT,
        #     "args": asdict(
        #         Ortools_Parameter(intersection=True, degree=True, maximize_edges=0.8)
        #     ),
        #     "hack_eval_6": True,
        #     "hack_eval_6_data": data,
        #     "hack_eval_6_PERCENT": 0.8,
        # },
        # {
        #     "timeout": TIMEOUT,
        #     "args": asdict(
        #         Ortools_Parameter(intersection=True, degree=True, maximize_edges=-0.1)
        #     ),
        #     "hack_eval_6": True,
        #     "hack_eval_6_data": data,
        #     "hack_eval_6_PERCENT": -0.1,
        # },
        # {
        #     "timeout": TIMEOUT,
        #     "args": asdict(
        #         Ortools_Parameter(intersection=True, degree=True, maximize_edges=-0.5)
        #     ),
        #     "hack_eval_6": True,
        #     "hack_eval_6_data": data,
        #     "hack_eval_6_PERCENT": -0.5,
        # },
        {
            "timeout": TIMEOUT,
            "args": asdict(
                Ortools_Parameter(intersection=True, degree=True, maximize_edges=-0.8)
            ),
            "hack_eval_6": True,
            "hack_eval_6_data": data,
            "hack_eval_6_PERCENT": -0.8,
        },
    ]
}


RI = Run_Algbench(
    inst_path=path,
    outer_parameter=outer_parameter,
    figure_path="/home/fabian/uni/Bachelorarbeit/bachelorarbeit-fabian-alich/code/evaluation/eval#6",
    path_benchmark="/home/fabian/uni/Bachelorarbeit/bachelorarbeit-fabian-alich/code/evaluation/benchmark",
    host=["algra01", "algra02", "algra03", "algra04", "algra05", "algra06"],
)


[15:27] INFO: Loaded data from /home/fabian/uni/Bachelorarbeit/bachelorarbeit-fabian-alich/code/evaluation/eval#6/calculated_data.json


In [5]:
RI.get_run_list()


['Ortools_delaunay_000_delaunay_30']

In [11]:
def run_solver_on_inst(key: str):
    solver, nodes, possible, inst, file_name = RI.get_solver_inst_from_runlist[key]
    parameters = RI.outer_parameter[solver]
    for parameter in parameters:
        ####################################################
        # hack für eval 6
        aktive_edges_percent = []
        not_aktive_edges_percent = []
        if parameter.get("hack_eval_6", False):
            try:
                if "hack_eval_6_data" not in parameter:
                    raise ValueError(
                        "hack_eval_6_data must be provided in the parameter."
                    )
                if "hack_eval_6_PERCENT" not in parameter:
                    raise ValueError(
                        "hack_eval_6_PERCENT must be provided in the parameter."
                    )
                data = parameter["hack_eval_6_data"]
                percent = parameter["hack_eval_6_PERCENT"]
                key = f"{inst}_{file_name}"
                if key not in data:
                    raise ValueError(f"No data found for instance {key}.")
                aktive_edges = data[key]
                all_edges = []
                for i in range(len(nodes)):
                    for j in range(i + 1, len(nodes)):
                        all_edges.append((nodes[i].pos, nodes[j].pos))

                not_aktive_edges = [
                    edge for edge in all_edges if edge not in aktive_edges
                ]
                # print("Aktive Edges:", len(aktive_edges))
                # print(*aktive_edges, sep="\n")
                # print("Nicht Aktive Edges:", len(not_aktive_edges))
                # print(*not_aktive_edges, sep="\n")
                # sys.exit(0)
                if percent > 0:
                    anzahl = max(1, int(len(aktive_edges) * percent))
                    auswahl = random.sample(aktive_edges, anzahl)
                    aktive_edges_percent = auswahl
                if percent < 0:
                    anzahl = max(1, int(len(not_aktive_edges) * -percent))
                    auswahl = random.sample(not_aktive_edges, anzahl)
                    not_aktive_edges_percent = auswahl

            except ValueError as e:
                logging.error(f"Error in hack_eval_6: {e}")
                continue
        ############################################

        for i in range(NUMBER_RUNS):
            run_seed = int(uuid.uuid4())
            random.seed(run_seed)  # Seed für Reproduzierbarkeit
            random.shuffle(nodes)  # Zufällige Reihenfolge der Knoten
            graph = Graph_Wrapper(nodes)
            pos_to_node_index = {
                get_key_from_pos(node.pos): i for i, node in enumerate(nodes)
            }  # Mapping von Position zu Knoten
            parameter["debug_set_edges"] = []
            for edge_pos in aktive_edges_percent:
                node1 = pos_to_node_index[get_key_from_pos(edge_pos[0])]
                node2 = pos_to_node_index[get_key_from_pos(edge_pos[1])]
                parameter["debug_set_edges"].append(
                    (min(node1, node2), max(node1, node2))
                )

            parameter["debug_exclude_edges"] = []
            for edge_pos in not_aktive_edges_percent:
                node1 = pos_to_node_index[get_key_from_pos(edge_pos[0])]
                node2 = pos_to_node_index[get_key_from_pos(edge_pos[1])]
                parameter["debug_exclude_edges"].append(
                    (min(node1, node2), max(node1, node2))
                )
        print(*aktive_edges, sep="\n")
        print("-" * 50)
        print(*not_aktive_edges, sep="\n")
        print("-" * 50)
        print(*parameter["debug_exclude_edges"], sep="\n")
run_solver_on_inst("Ortools_delaunay_000_delaunay_30")


((3007, 8515), (3444, 8522))
((3007, 8515), (2563, 9216))
((3007, 8515), (2398, 7227))
((3007, 8515), (2343, 9229))
((8561, 6473), (9211, 3427))
((8561, 6473), (7374, 6446))
((8561, 6473), (8063, 7170))
((4369, 3211), (1134, 4928))
((4369, 3211), (463, 1746))
((4369, 3211), (607, 1438))
((4369, 3211), (4613, 3179))
((4369, 3211), (4316, 4664))
((4369, 3211), (4917, 1249))
((4369, 3211), (3801, 84))
((1125, 6391), (1417, 7240))
((1125, 6391), (1134, 4928))
((1125, 6391), (274, 7351))
((1125, 6391), (115, 4108))
((1125, 6391), (2398, 7227))
((3444, 8522), (5812, 6288))
((3444, 8522), (4370, 8833))
((3444, 8522), (2563, 9216))
((3444, 8522), (2398, 7227))
((1417, 7240), (1302, 9633))
((1417, 7240), (274, 7351))
((1417, 7240), (2398, 7227))
((1417, 7240), (2343, 9229))
((9211, 3427), (8602, 1013))
((9211, 3427), (6370, 2734))
((9211, 3427), (6577, 2022))
((9211, 3427), (7867, 1256))
((9211, 3427), (7374, 6446))
((8602, 1013), (7867, 1256))
((8602, 1013), (3801, 84))
((6370, 2734), (6577, 2